# ASR — four variants, four regimes

ASR calibrates a clean-covariance baseline and reconstructs segments whose variance leaves that subspace. The variants differ in *where they look for clean reference data*.

*Deep dive behind the [five-minute demo](../meta_mne_denoise_demo.ipynb).*

## Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("demo_utils.py").exists():
    done = subprocess.run(
        ["git", "clone", "-q", "--depth", "1",
         "https://github.com/snesmaeili/mne-denoise-meta-demo.git"],
        capture_output=True, text=True)
    if done.returncode != 0:
        raise SystemExit(
            "Could not clone the demo repository. If it is still private, the Colab "
            "VM has no credentials for it -- authorising Colab lets it OPEN a "
            "notebook, not clone the repo. Make it public, or run locally."
        )
    os.chdir("mne-denoise-meta-demo")
    sys.path.insert(0, os.getcwd())
try:
    import mne_denoise  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "mne-denoise @ git+https://github.com/mne-tools/mne-denoise.git@f5b821cc2a535e84ed46085d45ea5a356dd8d548"],
                   check=True)

# Sections below read results prepared offline by prepare_meta_demo.py; the raw
# recordings are too large to ship with the notebook.
_cache = Path(os.environ.get("MNE_DENOISE_META_DEMO_CACHE",
                             Path.home() / ".cache" / "mne-denoise" / "meta-demo"))
if not (_cache / "dss_metrics.json").exists():
    subprocess.run([sys.executable, "fetch_demo_data.py"], check=False)

import warnings, logging
import numpy as np
import matplotlib.pyplot as plt
import mne

mne.set_log_level("ERROR")
logging.getLogger("mne_denoise").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*Epochs are not baseline corrected.*")
%matplotlib inline
RANDOM_STATE = 97

| Variant | Source | Built for |
|---|---|---|
| `ASR(method="standard")` | Kothe & Jung; Chang et al. 2020 | transient bursts on ordinary EEG |
| `ASR(method="riemannian_windowed")` | Blum et al. 2019 | calibration windows themselves contaminated |
| `AdaptiveASR(variant=...)` | Tsai et al. | non-stationary recordings, streaming BCI |
| `JugglerASR(strategy=...)` | Kim et al. 2025 | extreme MoBI — 205-channel juggling EEG |

`GuidedASR` also exists and is labelled in its own source as an unvalidated
research prototype. It is deliberately not demonstrated here.

## A fixture with known ground truth

In [ ]:
sys.path.insert(0, str(Path.cwd()))
from _asr_fixture import FixtureSpec, build_fixture, build_masks, rrmse, mean_channel_corr

spec = FixtureSpec(duration_s=60.0, random_state=RANDOM_STATE)
clean, contaminated, events, _pos, meta = build_fixture(spec)
artifact, guard, clean_mask = build_masks(events, spec.n_times, spec.sfreq)

# ASR documents that it expects high-pass filtered input.
clean = mne.filter.filter_data(clean, spec.sfreq, 1.0, None, verbose="ERROR")
contaminated = mne.filter.filter_data(contaminated, spec.sfreq, 1.0, None, verbose="ERROR")

print(f"{contaminated.shape[0]} channels, {spec.duration_s:.0f} s, {len(events)} events")
print(f"artifact {artifact.mean():.1%} | guard {guard.mean():.1%} | clean {clean_mask.mean():.1%}")

## Two disjoint endpoints

Removing artifact and preserving signal are separate questions, so they are scored on disjoint samples.

In [ ]:
from mne_denoise.asr import ASR, JugglerASR

results = {}
for label, est in [
    ("standard", ASR(sfreq=spec.sfreq)),
    ("riemannian_windowed", ASR(sfreq=spec.sfreq, method="riemannian_windowed")),
    ("juggler-gev", JugglerASR(sfreq=spec.sfreq, strategy="gev")),
]:
    out = np.asarray(est.fit_transform(contaminated.copy()))
    results[label] = dict(
        artifact=rrmse(out, clean, artifact),
        clean=rrmse(out, clean, clean_mask),
        corr=mean_channel_corr(out, clean, clean_mask)[0],
        est=est,
    )

print(f"{'variant':22s} {'artifact RRMSE':>15s} {'clean RRMSE':>12s} {'corr':>7s}")
print(f"{'contaminated':22s} {rrmse(contaminated, clean, artifact):15.4f} "
      f"{rrmse(contaminated, clean, clean_mask):12.4f} {'-':>7s}")
for k, v in results.items():
    print(f"{k:22s} {v['artifact']:15.4f} {v['clean']:12.4f} {v['corr']:7.3f}")

## Where did each one look for clean data?

In [ ]:
from mne_denoise.viz import plot_asr_calibration_fraction

ests = [v["est"] for v in results.values()]
fig, ax = plot_asr_calibration_fraction(ests, labels=list(results), show=False)
plt.show()

for k, v in results.items():
    ci = v["est"].calibration_info_
    if "reference_selected_fraction" in ci:
        print(f"  {k:22s} sample-based, kept {ci['reference_selected_fraction']:.1%}")
    else:
        print(f"  {k:22s} window-based, kept "
              f"{ci['n_clean_windows']}/{ci['n_calibration_windows']} windows")

## When and how much did it repair?

In [ ]:
from mne_denoise.viz import plot_asr_repair_timeline, plot_asr_component_reconstruction

est = results["standard"]["est"]
fig, ax = plot_asr_repair_timeline(est, show=False)
plt.show()
fig, ax = plot_asr_component_reconstruction(est, show=False)
plt.show()

> **Note.** At package defaults `standard` and `riemannian_windowed` can be numerically identical: both use `cov_estimator='geometric_median'`, and the only structural difference is partial-block handling. Blum's robustness lives in the covariance aggregator, which is already the default for every variant.

## Which variant does *this* recording need?

This section was cut from the five-minute talk for time; it is the question people actually ask. Two arms, both prepared offline by `prepare_meta_demo.py --asr-variants`:

- **A — a contaminated calibration period.** Same fixture, one declared change: artifacts are also present during the stretch ASR calibrates on. Does Blum's Riemannian option rescue it?
- **B — calibration supply.** On real mobile EEG, how much of the recording does each variant consider clean enough to calibrate on? Standard and rASR select clean *windows*; Juggler selects clean *samples*.

In [ ]:
import json

import demo_utils as du

V = json.loads((_cache / "asr_variants_metrics.json").read_text())
arm_a, arm_b = V["arm_a_contaminated_calibration"], V["arm_b_calibration_supply"]

with du.presentation_theme():
    fig = du.plot_asr_variant_regimes(arm_a, arm_b)
plt.show()

print(f"method='standard' and method='riemannian_windowed' identical here: "
      f"{arm_a['methods_identical']}")
print("   because both already default to cov_estimator='geometric_median'\n")
print(f"{arm_b['dataset']}")
for r in arm_b["rows"]:
    print(f"   {r['variant']:<15s} calibrates on {100 * r['calibration_fraction']:5.1f}% "
          f"of the recording ({r['calibration_kind']}-based), {r['runtime_s']:.1f} s")

> **How to read this.** Robustness to a contaminated calibration comes from `cov_estimator`, not from `method=`. On real mobile EEG the window selector is not starving, so Juggler is not indicated there — its variants exist for regimes where standard ASR refuses to calibrate at all.
>
> The practical rule: the fitted state answers "which variant" before you have to guess. Check `calibration_info_` first.